# Create Eval1 subset

This notebook creates a reproducible subset of 30 queries from the Biomni Eval1 dataset.

The goal is to define a fixed benchmark subset that can be reused across different agents and plataforms.

The generated CSV contains the input queries, task identifiers, and Eval1 reference answers.


## Output:
`project_subset.csv`

Main columns:
- `sample_index`
- `task_name`
- `task_instance_id`
- `input_query`
- `dataset_eval1_answer`

The `task_name` and `task_instance_id` columns are required later by the Biomni Eval1 scorer.


##### Imports

In [ ]:
import csv
import random
from pathlib import Path

import pandas as pd
from datasets import load_dataset

##### Auxiliary functions

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return text

    return (
        text.replace("\n", " ")
        .replace("\r", " ")
        .replace("\t", " ")
        .replace("  ", " ")
        .strip()
    )

##### Load Dataset

In [ ]:
dataset = load_dataset("biomni/Eval1", split="test")
rows = list(dataset)

print("Total rows:", len(rows))
print("Columns:", dataset.column_names)

##### Configuration

In [ ]:
sample_size = 30
seed = 42

output_path = Path("project_subset.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

##### Sample queries

In [ ]:
rng = random.Random(seed)
sampled_rows = rng.sample(rows, sample_size)

subset = []

for i, row in enumerate(sampled_rows, start=1):
    subset.append(
        {
            "sample_index": i,
            "task_name": row.get("task_name", ""),
            "task_instance_id": row.get("task_instance_id", row.get("instance_id", "")),
            "input_query": clean_text(row.get("prompt", "")),
            "dataset_eval1_answer": clean_text(row.get("answer", "")),
        }
    )

df = pd.DataFrame(subset)

df.head()

##### Save subset

In [ ]:
df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL,
)

print("Saved to:", output_path)
print("Rows:", len(df))
print("Columns:", df.columns.tolist())